# RSA on GPU (group steps 3, 5, 6, 7)

Takes the **per-participant** maps a previous Colab run already wrote to Drive
(`result_<model>_<specie>-sub-NN.zip`, from steps 1/2/4) and reduces them to the
group maps that pipeline steps 8-10 need:

| Step | What it produces |
|---|---|
| 3 | group mean/std of the real model-similarity maps |
| 5 | `reps_group` group permutation maps (one draw per participant each) |
| 6 | the voxelwise null distribution (mean/std across those) |
| 7 | a z map per permutation, plus the real z map |

Nothing has to be downloaded and re-uploaded in between: the participant maps are
read straight out of the result zips sitting in `RESULTS_DIR`.

**Runtime:** GPU (L4 or T4). Pick **High-RAM** for humans - the run holds
`reps_group x n_mask_voxels` float64 arrays.

**Steps:**
1. Upload the `pkg_group_*.zip` built by `tools/create_group_package.py` to Drive.
2. Set `PKG_ZIP`, `RESULTS_DIR` (where your `result_*.zip` live) and `OUT_DIR`.
3. Run all cells. One `result_group_<model>_<specie>.zip` per model appears in
   `OUT_DIR`; re-running skips models already done.
4. Back on the workstation: `tools/unpack_results.py` merges them onto the pipeline
   disk, then run steps 8-10 as usual.

In [ ]:
# 1. Check the GPU and install nibabel (torch is preinstalled on Colab).
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
!pip -q install nibabel

In [ ]:
# 2. Mount Google Drive.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 3. EDIT THESE.
PKG_ZIP     = '/content/drive/MyDrive/rsa_colab/pkg_group_D_EmoC_basic-block_mahalanobis.zip'
RESULTS_DIR = '/content/drive/MyDrive/rsa_colab/results'   # where result_<model>_<specie>-sub-NN.zip live
OUT_DIR     = '/content/drive/MyDrive/rsa_colab/results'   # where result_group_*.zip go (can be the same)

STEPS       = [3, 5, 6, 7]  # step 7's real z map needs step 3's group mean
BATCH       = 20000         # voxels per GPU chunk; lower if you hit out-of-memory
G_BATCH     = 64            # group permutations gathered per pass
WORKERS     = 8             # threads for reading result zips / writing niftis

# Step 5's reps_group group mean maps are inputs to steps 6 and 7 only, and both
# run here. Set this False to leave them out and roughly halve what you download;
# steps 8-10 only ever read the z maps.
WRITE_GROUP_MEANS = True

In [ ]:
# 4. Unzip the group package into Colab-local storage.
import os, zipfile, shutil
PKG_ROOT = '/content/pkg_group'
if os.path.isdir(PKG_ROOT):
    shutil.rmtree(PKG_ROOT)
os.makedirs(PKG_ROOT, exist_ok=True)
with zipfile.ZipFile(PKG_ZIP) as zf:
    zf.extractall(PKG_ROOT)
print('unpacked to', PKG_ROOT)
print(sorted(os.listdir(PKG_ROOT)))

In [ ]:
# 5. Sanity check: how many participant result zips can we see?
import sys, json
sys.path.insert(0, os.path.join(PKG_ROOT, 'code'))
import gpu_group

manifest = gpu_group.load_manifest(PKG_ROOT)
store = gpu_group.ResultStore([RESULTS_DIR], dataset=manifest['dataset'])
print(f"{manifest['specie']}  {manifest['dataset']}/{manifest['model']}  "
      f"reps={manifest['reps']}  reps_group={manifest['reps_group']}")
print(f"participants: {len(manifest['participants'])}   models: {len(manifest['models'])}")
for m in manifest['models'][:3]:
    print(f"  {m}: {len(store.zips_for(manifest, m))} participant zip(s)")

In [ ]:
# 6. Run the group steps for every model. Resumable: skips models already done.
import run_colab_group
written = run_colab_group.run_group_package(
    PKG_ROOT, RESULTS_DIR, OUT_DIR,
    work_root='/content/group_work',
    steps=STEPS, batch=BATCH, g_batch=G_BATCH, workers=WORKERS,
    write_group_means=WRITE_GROUP_MEANS, verbose=True)
print('\nnew group result zips:')
for w in written:
    print(' ', w)